# Appliance Energy Forecasting — Part 7 & 8: Foundation Model & Full Comparison

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = "/content/drive/MyDrive/appliance-energy-forecasting"
os.chdir(PROJECT_ROOT)


In [ ]:
!pip uninstall -y torchvision -q
!pip install -q chronos-forecasting

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error
from chronos import ChronosPipeline

plt.rcParams["figure.figsize"] = (14, 5)
np.random.seed(0)

os.makedirs("outputs/figures", exist_ok=True)
os.makedirs("outputs/forecasts", exist_ok=True)
os.makedirs("outputs/metrics", exist_ok=True)


In [ ]:
hourly = pd.read_csv("data/processed/appliance_hourly.csv", index_col=0, parse_dates=True)
y = hourly["Appliances"]

TARGET = "Appliances"
HORIZON = 24
DAILY_PERIOD = 24
TEST_STEPS = 14 * 24

train = y.iloc[:-TEST_STEPS]
test = y.iloc[-TEST_STEPS:]


In [ ]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def mase(y_true, y_pred, y_train, seasonality=24):
    y_train = pd.Series(y_train).astype(float)
    seasonal_errors = np.abs(y_train.iloc[seasonality:].values - y_train.iloc[:-seasonality].values)
    scale = seasonal_errors.mean()
    if scale == 0:
        return np.nan
    return np.mean(np.abs(y_true - y_pred)) / scale

def evaluate_forecast(name, y_true, y_pred, y_train):
    y_true = pd.Series(y_true).astype(float)
    y_pred = pd.Series(y_pred, index=y_true.index).astype(float)
    return {
        "model": name,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": rmse(y_true, y_pred),
        "MASE": mase(y_true, y_pred, y_train, seasonality=DAILY_PERIOD),
        "Bias": np.mean(y_pred - y_true),
    }


### Load Chronos (zero-shot, target-only)

`chronos-t5-small` is used for speed on Colab. Chronos is used **zero-shot** here — no
fine-tuning — and target-only, since the base Chronos model does not natively support
exogenous covariates. This is an important limitation to discuss in the report (Part 9,
question 4): the SARIMAX and feature-based models had access to weather/time covariates
that Chronos does not use here.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("using device:", device)

pipeline = ChronosPipeline.from_pretrained(
    "amazon/chronos-t5-small",
    device_map=device,
    torch_dtype=torch.float32,
)


### Rolling 24h zero-shot forecast

Chronos takes the recent history as context and forecasts the next `HORIZON` steps directly,
without being retrained. A rolling context window (last 512 hours) is used to keep inference
fast while still giving the model several weeks of recent context.

In [ ]:
CONTEXT_LENGTH = 512

def rolling_chronos_forecast(pipeline, y_train, y_test, horizon=24, context_length=512):
    history = y_train.copy()
    preds = []

    for start in range(0, len(y_test), horizon):
        block_index = y_test.index[start:start + horizon]
        block_horizon = len(block_index)

        context = torch.tensor(history.iloc[-context_length:].values, dtype=torch.float32)
        forecast = pipeline.predict(context, prediction_length=block_horizon, num_samples=20)

        # median across samples as point forecast
        median_pred = np.median(forecast[0].numpy(), axis=0)
        preds.append(pd.Series(median_pred, index=block_index))

        history = pd.concat([history, y_test.iloc[start:start + block_horizon]])

    return pd.concat(preds)

chronos_pred = rolling_chronos_forecast(pipeline, train, test, horizon=HORIZON, context_length=CONTEXT_LENGTH)


In [ ]:
chronos_results = evaluate_forecast("foundation_chronos", test, chronos_pred.reindex(test.index), train)
print(chronos_results)


In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
plot_window = test.index[:24*7]

test.loc[plot_window].plot(ax=ax, label="actual", color="black", linewidth=2)
chronos_pred.reindex(plot_window).plot(ax=ax, label="chronos forecast", color="tab:purple")
ax.set_title("Chronos Foundation Model Forecast vs Actual - First 7 Days of Test Period")
ax.set_ylabel("Appliances (Wh)")
ax.legend()
plt.tight_layout()
plt.savefig("outputs/figures/11_chronos_forecast.png", dpi=150)
plt.show()


In [ ]:
chronos_forecast_df = pd.DataFrame({"actual": test, "foundation_chronos": chronos_pred.reindex(test.index)})
chronos_forecast_df.to_csv("outputs/forecasts/chronos_forecast.csv")
pd.DataFrame([chronos_results]).to_csv("outputs/metrics/chronos_metrics.csv", index=False)
print("saved chronos forecast and metrics")


## Part 8: Full model comparison

Loads every forecast and metric saved by notebooks 2-5 and combines them into one table
and one plot, so every model is compared against the same test period and against the
strongest benchmark.

In [ ]:
benchmark_forecasts = pd.read_csv("outputs/forecasts/benchmark_forecasts.csv", index_col=0, parse_dates=True)
sarimax_forecasts = pd.read_csv("outputs/forecasts/sarimax_forecast.csv", index_col=0, parse_dates=True)
feature_forecasts = pd.read_csv("outputs/forecasts/feature_model_forecast.csv", index_col=0, parse_dates=True)
chronos_forecasts = pd.read_csv("outputs/forecasts/chronos_forecast.csv", index_col=0, parse_dates=True)

all_forecasts = benchmark_forecasts.copy()
all_forecasts["sarimax"] = sarimax_forecasts["sarimax"]
all_forecasts["feature_model"] = feature_forecasts["feature_model"]
all_forecasts["foundation_chronos"] = chronos_forecasts["foundation_chronos"]

all_forecasts.to_csv("outputs/forecasts/all_forecasts.csv")
all_forecasts.head()


In [ ]:
benchmark_metrics = pd.read_csv("outputs/metrics/benchmark_comparison.csv")
sarimax_metrics = pd.read_csv("outputs/metrics/sarimax_metrics.csv")
feature_metrics = pd.read_csv("outputs/metrics/feature_model_metrics.csv")
chronos_metrics = pd.read_csv("outputs/metrics/chronos_metrics.csv")

all_metrics = pd.concat([benchmark_metrics, sarimax_metrics, feature_metrics, chronos_metrics], ignore_index=True)
all_metrics = all_metrics.sort_values("MASE").reset_index(drop=True)

all_metrics.to_csv("outputs/metrics/model_comparison.csv", index=False)
print(all_metrics.round(3))


### Identify strongest benchmark and compare every model against it

The assignment explicitly requires comparing all models against the strongest benchmark,
not only against each other.

In [ ]:
benchmark_names = ["mean", "naive", "seasonal_naive_daily", "seasonal_naive_weekly", "drift"]
strongest_benchmark_row = all_metrics[all_metrics["model"].isin(benchmark_names)].sort_values("MASE").iloc[0]
strongest_benchmark = strongest_benchmark_row["model"]
print("strongest benchmark:", strongest_benchmark)

all_metrics["improvement_vs_strongest_benchmark_pct"] = (
    (strongest_benchmark_row["MASE"] - all_metrics["MASE"]) / strongest_benchmark_row["MASE"] * 100
)
print(all_metrics[["model", "MASE", "improvement_vs_strongest_benchmark_pct"]].round(2))

all_metrics.to_csv("outputs/metrics/model_comparison.csv", index=False)


In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))
plot_window = test.index[:24*7]

all_forecasts.loc[plot_window, "actual"].plot(ax=ax, label="actual", color="black", linewidth=2.5)
for col in all_forecasts.columns:
    if col != "actual":
        all_forecasts.loc[plot_window, col].plot(ax=ax, label=col, alpha=0.75)

ax.set_title("All Models: Forecast Comparison - First 7 Days of Test Period")
ax.set_ylabel("Appliances (Wh)")
ax.set_xlabel("Date")
ax.legend(ncol=2)
plt.tight_layout()
plt.savefig("outputs/figures/12_all_models_comparison.png", dpi=150)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
plot_metrics = all_metrics.sort_values("MASE")
ax.barh(plot_metrics["model"], plot_metrics["MASE"])
ax.invert_yaxis()
ax.axvline(1.0, color="red", linestyle="--", label="MASE = 1 (naive seasonal baseline)")
ax.set_xlabel("MASE (lower is better)")
ax.set_title("Model Comparison by MASE")
ax.legend()
plt.tight_layout()
plt.savefig("outputs/figures/13_mase_comparison.png", dpi=150)
plt.show()
